# Preprocessing — Jalur **IndoBERT** (semua versi)

**Tujuan.** Memproses **seluruh 10.000 komentar berlabel** dengan pembersihan minimal
dan menyimpan ke koleksi **`processed_bert`** beserta **flag 4 versi**
(`in_set6k`, `in_balanced_set`, `in_set10k`, `in_balanced10k`).

Sama seperti jalur SVM, preprocessing tidak bergantung versi → diproses sekali; split &
pemilihan versi dilakukan di notebook fine-tuning. Pembersihan **minimal**
(`clean_for_bert`): lowercase + buang URL/mention/emoji; tanda baca, angka, imbuhan, dan
negasi dipertahankan untuk tokenizer IndoBERT.

## 0. Dependency

In [1]:
%pip install -q "pymongo[srv]" dnspython certifi python-dotenv scikit-learn pandas numpy

Note: you may need to restart the kernel to use updated packages.


## 1. Baca semua data berlabel (10k) + flag versi

In [2]:
import os, pandas as pd
from pymongo import MongoClient
import certifi
MONGO_URI=os.environ.get("MONGO_URI","")
if not MONGO_URI:
    try:
        from dotenv import load_dotenv; load_dotenv(); MONGO_URI=os.environ.get("MONGO_URI","")
    except Exception: pass
if not MONGO_URI:
    from getpass import getpass; MONGO_URI=getpass("MONGO_URI: ")
DB=os.environ.get("MONGO_DB_NAME","youtube_sentiment")
client=MongoClient(MONGO_URI,tlsCAFile=certifi.where(),serverSelectionTimeoutMS=20000)
client.admin.command("ping"); print("Koneksi MongoDB OK.")
LABELS=["Negatif","Netral","Positif"]
FLAGS=["in_set6k","in_balanced_set","in_set10k","in_balanced10k"]
proj={"_id":0,"comment_id":1,"video_id":1,"text":1,"label":1}
for fl in FLAGS: proj[fl]=1
# Ambil SEMUA komentar berlabel (10k) — preprocessing tak tergantung versi.
df=pd.DataFrame(list(client[DB]["raw_comments"].find({"label":{"$exists":True}},proj)))
for fl in FLAGS: df[fl]=df[fl].fillna(False)
print(f"{len(df)} dok berlabel (semua versi) dari Mongo")
print("Jumlah per versi:", {fl:int(df[fl].sum()) for fl in FLAGS})

Koneksi MongoDB OK.


10000 dok berlabel (semua versi) dari Mongo
Jumlah per versi: {'in_set6k': 6000, 'in_balanced_set': 3000, 'in_set10k': 10000, 'in_balanced10k': 5808}


## 2. Fungsi preprocessing (inline)

In [3]:
import re

_RE_URL = re.compile('https?://\\S+|www\\.\\S+', re.IGNORECASE)
_RE_MENTION = re.compile('@\\w+')
_RE_EMOJI = re.compile('[😀-🙏🌀-🗿🚀-\U0001f6ff🜀-🝿🞀-\U0001f7ff🠀-\U0001f8ff🤀-🧿🨀-\U0001fa6f🩰-\U0001faff✂-➰Ⓜ-🉑🤦-🤷\u200d♀-♂☀-⭕⏏⏩⌚️〰]+', re.UNICODE)
_RE_MULTI_SPACE = re.compile('\\s+')

def clean_for_bert(text: str) -> str:
    """Cleaning minimal untuk jalur IndoBERT (morfologi terjaga)."""
    if not text or not isinstance(text, str):
        return ""
    t = text.lower()
    t = _RE_URL.sub(" ", t)
    t = _RE_MENTION.sub(" ", t)
    t = _RE_EMOJI.sub(" ", t)
    t = re.sub(r"[^\w\s.,!?;:'\"()-]", " ", t)
    t = _RE_MULTI_SPACE.sub(" ", t)
    return t.strip()


def make_text_bert(text: str) -> str:
    return clean_for_bert(text or "")

## 3. Transformasi (satu tahap)

In [4]:
def trace_bert(t):
    print(f"RAW              : {t!r}"); print(f"clean_for_bert   : {clean_for_bert(t)!r}   <- FINAL")
trace_bert("Ijazahnya tidak palsu kok, jgn asal tuduh tanpa bukti")

RAW              : 'Ijazahnya tidak palsu kok, jgn asal tuduh tanpa bukti'
clean_for_bert   : 'ijazahnya tidak palsu kok, jgn asal tuduh tanpa bukti'   <- FINAL


### Tabel before→after (sampel acak)

In [5]:
import pandas as pd
pd.set_option("display.max_colwidth",55)
_s=df.sample(6,random_state=7).copy(); _s["bert"]=_s["text"].astype(str).map(make_text_bert)
_s[["label","text","bert"]]

,label,text,bert
1977,Negatif,Susah emang orang seperti itu..dulu koar koar ayo ...,susah emang orang seperti itu..dulu koar koar ayo t...
3880,Negatif,kelompok orang2 sakit hati yg memburu ijajah pak jo...,kelompok orang2 sakit hati yg memburu ijajah pak jo...
52,Positif,Kasihan ya.. PEMBICARANYA SEPERTI MENANGGUNG BEBAN ...,kasihan ya.. pembicaranya seperti menanggung beban ...
2551,Positif,JKW dirumah liat ini kejang2,jkw dirumah liat ini kejang2
2246,Negatif,Kalau hati sudah busuk apapun yang kita lakukan tid...,kalau hati sudah busuk apapun yang kita lakukan tid...
270,Positif,"Yg berbohong masalah ijazah jokowi, pasti segera di...","yg berbohong masalah ijazah jokowi, pasti segera di..."


## 4. Terapkan ke seluruh 10k

In [6]:
LABEL2ID={l:i for i,l in enumerate(LABELS)}
out_df=df.copy(); out_df["label_id"]=out_df["label"].map(LABEL2ID)
out_df["bert"]=out_df["text"].astype(str).map(make_text_bert)
before=len(out_df); out_df=out_df[out_df["bert"].str.len()>0].reset_index(drop=True)
print(f"{before-len(out_df)} baris dibuang (kosong); sisa {len(out_df)}")

0 baris dibuang (kosong); sisa 10000


## 5. EDA & ringkasan kualitas

In [7]:
# === EDA & ringkasan kualitas (seluruh 10k) ===
import numpy as np
print("Distribusi kelas (10k):"); print(out_df["label"].value_counts().reindex(LABELS).to_string())
wl_o=out_df["text"].astype(str).str.split().map(len); wl_p=out_df["bert"].astype(str).str.split().map(len)
print(f"\nPanjang(kata) ORIG: mean {wl_o.mean():.1f} median {wl_o.median():.0f} p95 {wl_o.quantile(.95):.0f} max {wl_o.max()}")
print(f"Panjang(kata) BERT : mean {wl_p.mean():.1f} median {wl_p.median():.0f} p95 {wl_p.quantile(.95):.0f} max {wl_p.max()}")
vocab=set(t for s in out_df["bert"].astype(str) for t in s.split())
print(f"\nVocab unik: {len(vocab)} | <=1 kata: {(wl_p<=1).sum()} | duplikat teks: {out_df['bert'].duplicated().sum()}")
print("\nDistribusi kelas per versi:")
for fl in FLAGS:
    sub=out_df[out_df[fl]]
    print(f"  {fl:<16} n={len(sub):<5} " + " ".join(f"{k}={v}" for k,v in sub['label'].value_counts().reindex(LABELS).items()))

Distribusi kelas (10k):
label
Negatif    4100
Netral     1936
Positif    3964

Panjang(kata) ORIG: mean 15.9 median 11 p95 42 max 689
Panjang(kata) BERT : mean 15.8 median 11 p95 41 max 681

Vocab unik: 25777 | <=1 kata: 4 | duplikat teks: 36

Distribusi kelas per versi:
  in_set6k         n=6000  Negatif=2622 Netral=1002 Positif=2376
  in_balanced_set  n=3000  Negatif=1000 Netral=1000 Positif=1000
  in_set10k        n=10000 Negatif=4100 Netral=1936 Positif=3964
  in_balanced10k   n=5808  Negatif=1936 Netral=1936 Positif=1936


## 6. Top term diskriminatif per kelas

In [8]:
# === Top term diskriminatif per kelas (skor lift, seluruh 10k) ===
from collections import Counter
cnt={l:Counter() for l in LABELS}; tot=Counter(); nclass={}
for l in LABELS:
    sub=out_df[out_df["label"]==l]; nclass[l]=len(sub)
    for s in sub["bert"].astype(str):
        for t in set(s.split()): cnt[l][t]+=1; tot[t]+=1
N=len(out_df)
def top(l,k=12,minc=8):
    o=[(t,round((c/nclass[l])/(tot[t]/N),2)) for t,c in cnt[l].items() if tot[t]>=minc]
    return sorted(o,key=lambda x:-x[1])[:k]
for l in LABELS: print(f"[{l}] "+", ".join(f"{t}({v})" for t,v in top(l)))

[Negatif] pembenci(2.44), panci,(2.44), nusa(2.44), dengki(2.44), nusakambangan(2.44), iri(2.34), stress(2.3), panci.(2.26), laku(2.24), jeruji(2.24), soryo(2.24), setres(2.24)
[Netral] bekerja(3.87), putusan(3.76), kemajuan(3.62), desa(3.44), siapa?(3.23), pembelaan(3.23), capres(3.23), nanti.(3.23), saling(3.19), bahas(3.13), hotman(3.13), bencana(3.05)
[Positif] ditutupi(2.52), klarifikasi(2.52), ijazahnya.(2.52), dugaan(2.52), kejanggalan(2.52), mengungkap(2.52), pramuka(2.52), istiqomah(2.52), palsunya(2.38), ova(2.37), kepalsuan(2.34), paksa(2.31)


## 7. Tulis ke koleksi `processed_bert` (dengan flag versi, tanpa split)

In [9]:
from pymongo import UpdateOne
OUT=client[DB]["processed_bert"]; ops=[]
for r in out_df.to_dict("records"):
    doc={"comment_id":r["comment_id"],"video_id":r.get("video_id"),"text":r["text"],
         "bert":r["bert"],"label":r["label"],"label_id":int(r["label_id"])}
    for fl in FLAGS: doc[fl]=bool(r[fl])
    ops.append(UpdateOne({"comment_id":r["comment_id"]},{"$set":doc},upsert=True))
OUT.delete_many({})              # bersihkan isi lama (dulu cuma v2 + ada field split)
OUT.bulk_write(ops,ordered=False)
print("processed_bert:",OUT.count_documents({}),"dok (semua versi, tanpa split)")

processed_bert: 10000 dok (semua versi, tanpa split)


✅ **Selesai.** `processed_bert` berisi 10k teks + flag versi. Split & pemilihan versi di **`indobert_finetune_colab.ipynb`**.